In [1]:
import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score
from sklearn.metrics import roc_auc_score

from metric import precision_at_recall

In [2]:
train = pd.read_csv("data/train.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
test = pd.read_csv("data/test.csv", parse_dates=["cookie_created_at", "window_start_ts", "window_end_ts"])
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

In [3]:
def events_in_window(events, meta):
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id", how="inner")
    ev = ev[(ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])].copy()
    ev = ev.drop(columns=["window_start_ts", "window_end_ts"])
    return ev

In [4]:
ev_train = events_in_window(events, train)
ev_test = events_in_window(events, test)


In [5]:
def make_meta_features(meta):
    f = meta[["cookie_id"]].copy()
    f["cookie_age_hours"] = (meta["window_start_ts"] - meta["cookie_created_at"]).dt.total_seconds() / 3600
    f["log_cookie_age_hours"] = np.log1p(f["cookie_age_hours"].clip(lower=0))
    return f

In [6]:
def make_basic_features(ev, meta):
    g = ev.groupby("cookie_id")
    f = pd.DataFrame(index=g.size().index)
    f["n_events"] = g.size()
    f["event_type_nunique"] = g["event_name"].nunique()
    f["item_nunique"] = g["item_id"].nunique()
    f["category_nunique"] = g["item_category"].nunique()
    f["location_nunique"] = g["item_location"].nunique()
    f["seller_type_nunique"] = g["seller_type"].nunique()
    f = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [7]:
def make_time_features(ev, meta):
    x = ev[["cookie_id", "event_ts"]].copy()
    x = x.sort_values(["cookie_id", "event_ts"])
    x["gap_sec"] = x.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    x["gap_le_5"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 5, np.nan)
    x["gap_le_10"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 10, np.nan)
    x["gap_le_30"] = np.where(x["gap_sec"].notna(), x["gap_sec"] <= 30, np.nan)
    g = x.groupby("cookie_id")
    f = g.agg(first_event_ts=("event_ts", "min"), last_event_ts=("event_ts", "max"), mean_gap_sec=("gap_sec", "mean"), median_gap_sec=("gap_sec", "median"), std_gap_sec=("gap_sec", "std"), min_gap_sec=("gap_sec", "min"), max_gap_sec=("gap_sec", "max"), share_gap_le_5=("gap_le_5", "mean"), share_gap_le_10=("gap_le_10", "mean"), share_gap_le_30=("gap_le_30", "mean")).reset_index()
    f["active_span_sec"] = (f["last_event_ts"] - f["first_event_ts"]).dt.total_seconds()
    f = f.drop(columns=["first_event_ts", "last_event_ts"])
    event_count = ev.groupby("cookie_id").size().rename("n_events_tmp").reset_index()
    f = f.merge(event_count, on="cookie_id", how="left")
    f["events_per_active_hour"] = np.where(f["active_span_sec"] > 0, (f["n_events_tmp"] - 1) / (f["active_span_sec"] / 3600), 0)
    f = f.drop(columns=["n_events_tmp"])
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [8]:
def make_event_count_features(ev, meta):
    counts = ev.groupby(["cookie_id", "event_name"]).size().unstack(fill_value=0)
    counts = counts.add_prefix("count_")
    f = meta[["cookie_id"]].merge(counts.reset_index(), on="cookie_id", how="left")
    f = f.fillna(0)
    count_cols = [col for col in f.columns if col.startswith("count_")]
    f["event_count_total"] = f[count_cols].sum(axis=1)

    for col in count_cols:
        f["share_" + col[6:]] = np.where(f["event_count_total"] > 0, f[col] / f["event_count_total"], 0)

    f = f.drop(columns=["event_count_total"])
    return f

In [9]:
def make_item_features(ev, meta):
    item_view = ev[ev["event_name"] == "item_view"].copy()
    f = item_view.groupby("cookie_id").agg(item_view_count=("item_id", "size"), item_view_nunique=("item_id", "nunique")).reset_index()
    f["item_view_repeat_count"] = f["item_view_count"] - f["item_view_nunique"]
    f["item_view_unique_ratio"] = np.where(f["item_view_count"] > 0, f["item_view_nunique"] / f["item_view_count"], 0)
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [10]:
def make_search_features(ev, meta):
    search = ev[ev["event_name"] == "search_results_view"].copy()
    f = search.groupby("cookie_id").agg(search_query_nunique=("search_query", "nunique"), search_page_mean=("search_page", "mean"), search_page_median=("search_page", "median"), search_page_max=("search_page", "max"), search_page_nunique=("search_page", "nunique")).reset_index()
    search_count = search.groupby("cookie_id").size().rename("search_event_count").reset_index()
    f = f.merge(search_count, on="cookie_id", how="outer")
    f["searches_per_query"] = np.where(f["search_query_nunique"] > 0, f["search_event_count"] / f["search_query_nunique"], 0)
    search["page_gt_1"] = search["search_page"] > 1
    search["page_gt_3"] = search["search_page"] > 3
    search["page_gt_5"] = search["search_page"] > 5
    depth = search.groupby("cookie_id").agg(share_page_gt_1=("page_gt_1", "mean"), share_page_gt_3=("page_gt_3", "mean"), share_page_gt_5=("page_gt_5", "mean")).reset_index()
    f = f.merge(depth, on="cookie_id", how="outer")
    query_page = search.dropna(subset=["search_query", "search_page"]).drop_duplicates(["cookie_id", "search_query", "search_page"])
    query_page_nunique = query_page.groupby("cookie_id").size().rename("query_page_nunique").reset_index()
    f = f.merge(query_page_nunique, on="cookie_id", how="outer")
    pages_per_query = query_page.groupby(["cookie_id", "search_query"]).size().rename("pages_for_query").reset_index()
    pages_summary = pages_per_query.groupby("cookie_id")["pages_for_query"].agg(["mean", "max"]).reset_index()
    pages_summary.columns = ["cookie_id", "pages_per_query_mean", "pages_per_query_max"]
    f = f.merge(pages_summary, on="cookie_id", how="outer")
    f["unique_query_page_ratio"] = np.where(f["search_event_count"] > 0, f["query_page_nunique"] / f["search_event_count"], 0)
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [11]:
def make_tech_features(ev, meta):
    x = ev[["cookie_id", "platform", "user_agent"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["ua_lower"] = x["user_agent"].str.lower()
    x["is_headless"] = x["ua_lower"].str.contains("headlesschrome")
    x["is_firefox"] = x["ua_lower"].str.contains("firefox")
    x["is_yandex"] = x["ua_lower"].str.contains("yabrowser")
    x["is_avito_app"] = x["ua_lower"].str.startswith("avito/")
    x["is_chrome"] = x["ua_lower"].str.contains("chrome") & ~x["is_headless"] & ~x["is_yandex"]
    x["is_windows"] = x["ua_lower"].str.contains("windows")
    x["is_linux"] = x["ua_lower"].str.contains("linux") & ~x["ua_lower"].str.contains("android")
    x["is_macos"] = x["ua_lower"].str.contains("macintosh")
    x["is_android_ua"] = x["ua_lower"].str.contains("android")
    x["ua_length"] = x["user_agent"].str.len()
    f = x.groupby("cookie_id").agg(platform_nunique=("platform_norm", "nunique"), ua_nunique=("user_agent", "nunique"), ua_length_mean=("ua_length", "mean"), ua_length_min=("ua_length", "min"), ua_length_max=("ua_length", "max"), is_headless=("is_headless", "max"), is_firefox=("is_firefox", "max"), is_yandex=("is_yandex", "max"), is_avito_app=("is_avito_app", "max"), is_chrome=("is_chrome", "max"), is_windows=("is_windows", "max"), is_linux=("is_linux", "max"), is_macos=("is_macos", "max"), is_android_ua=("is_android_ua", "max")).reset_index()
    platform_counts = x.groupby(["cookie_id", "platform_norm"]).size().unstack(fill_value=0)
    platform_counts = platform_counts.div(platform_counts.sum(axis=1), axis=0)
    platform_counts = platform_counts.add_prefix("platform_share_").reset_index()
    f = f.merge(platform_counts, on="cookie_id", how="left")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [12]:
def make_pointer_features(ev, meta):
    x = ev[["cookie_id", "event_ts", "platform", "pointer_x", "pointer_y"]].copy()
    x["platform_norm"] = x["platform"].astype("string").str.strip().str.lower()
    x["has_pointer"] = x["pointer_x"].notna() & x["pointer_y"].notna()
    basic = x.groupby("cookie_id").agg(pointer_event_count=("has_pointer", "sum"), pointer_event_share=("has_pointer", "mean")).reset_index()
    basic["has_any_pointer"] = basic["pointer_event_count"] > 0
    valid = x[x["has_pointer"]].copy()
    coord = valid.groupby("cookie_id").agg(pointer_x_mean=("pointer_x", "mean"), pointer_x_std=("pointer_x", "std"), pointer_x_min=("pointer_x", "min"), pointer_x_max=("pointer_x", "max"), pointer_y_mean=("pointer_y", "mean"), pointer_y_std=("pointer_y", "std"), pointer_y_min=("pointer_y", "min"), pointer_y_max=("pointer_y", "max")).reset_index()
    coord["pointer_x_range"] = coord["pointer_x_max"] - coord["pointer_x_min"]
    coord["pointer_y_range"] = coord["pointer_y_max"] - coord["pointer_y_min"]
    valid["_row_order"] = np.arange(len(valid))
    valid = valid.sort_values(["cookie_id", "event_ts", "_row_order"])
    valid["prev_x"] = valid.groupby("cookie_id")["pointer_x"].shift()
    valid["prev_y"] = valid.groupby("cookie_id")["pointer_y"].shift()
    valid["dx"] = valid["pointer_x"] - valid["prev_x"]
    valid["dy"] = valid["pointer_y"] - valid["prev_y"]
    valid["move_distance"] = np.sqrt(valid["dx"] ** 2 + valid["dy"] ** 2)
    moves = valid.groupby("cookie_id").agg(move_distance_mean=("move_distance", "mean"), move_distance_median=("move_distance", "median"), move_distance_std=("move_distance", "std"), move_distance_max=("move_distance", "max")).reset_index()
    pairs = valid.drop_duplicates(["cookie_id", "pointer_x", "pointer_y"]).groupby("cookie_id").size().rename("pointer_pair_nunique").reset_index()
    counts = valid.groupby("cookie_id").size().rename("pointer_count").reset_index()
    repeat = counts.merge(pairs, on="cookie_id", how="left")
    repeat["pointer_unique_ratio"] = np.where(repeat["pointer_count"] > 0, repeat["pointer_pair_nunique"] / repeat["pointer_count"], 0)
    platform_pointer = x.groupby(["cookie_id", "platform_norm"])["has_pointer"].mean().unstack(fill_value=0)
    platform_pointer = platform_pointer.add_prefix("pointer_share_").reset_index()
    f = basic.merge(coord, on="cookie_id", how="left")
    f = f.merge(moves, on="cookie_id", how="left")
    f = f.merge(repeat[["cookie_id", "pointer_unique_ratio"]], on="cookie_id", how="left")
    f = f.merge(platform_pointer, on="cookie_id", how="left")
    f = meta[["cookie_id"]].merge(f, on="cookie_id", how="left")
    f = f.fillna(0)
    return f

In [13]:
def merge_feature_groups(meta, feature_groups):
    result = meta[["cookie_id"]].copy()

    for features in feature_groups:
        result = result.merge(features, on="cookie_id", how="left")

    return result

In [14]:
basic_train_features = make_basic_features(ev_train, train)
basic_test_features = make_basic_features(ev_test, test)

meta_train_features = make_meta_features(train)
meta_test_features = make_meta_features(test)

time_train_features = make_time_features(ev_train, train)
time_test_features = make_time_features(ev_test, test)

event_train_features = make_event_count_features(ev_train, train)
event_test_features = make_event_count_features(ev_test, test)

tech_train_features = make_tech_features(ev_train, train)
tech_test_features = make_tech_features(ev_test, test)

pointer_train_features = make_pointer_features(ev_train, train)
pointer_test_features = make_pointer_features(ev_test, test)

item_train_features = make_item_features(ev_train, train)
item_test_features = make_item_features(ev_test, test)

search_train_features = make_search_features(ev_train, train)
search_test_features = make_search_features(ev_test, test)

In [15]:
train_features = merge_feature_groups(train, [basic_train_features, meta_train_features, time_train_features, event_train_features, tech_train_features, pointer_train_features, item_train_features, search_train_features])
test_features = merge_feature_groups(test, [basic_test_features, meta_test_features, time_test_features, event_test_features, tech_test_features, pointer_test_features, item_test_features, search_test_features])

In [16]:
temporal_folds = [("2026-04-11", "2026-04-13"), ("2026-04-14", "2026-04-16"), ("2026-04-17", "2026-04-19")]

In [17]:
def evaluate_model_temporal(features, model_factory, name):
    cols = [col for col in features.columns if col != "cookie_id"]
    y = train["target"].values
    dates = train["window_start_ts"]
    fold_results = []

    for fold_id, (valid_start, valid_end) in enumerate(temporal_folds, start=1):
        train_mask = dates < valid_start
        valid_mask = dates.between(valid_start, valid_end)
        model = model_factory()
        model.fit(features.loc[train_mask, cols], y[train_mask])
        pred = model.predict_proba(features.loc[valid_mask, cols])[:, 1]
        score = precision_at_recall(y[valid_mask], pred)
        roc_auc = roc_auc_score(y[valid_mask], pred)
        pr_auc = average_precision_score(y[valid_mask], pred)
        fold_results.append({"name": name, "fold": fold_id, "precision_at_recall_70": score, "roc_auc": roc_auc, "pr_auc": pr_auc})

    return pd.DataFrame(fold_results)

# Финальное обучение

После выбора модели обучим CatBoost и LightGBM на всей доступной обучающей выборке.

Сначала подготовим матрицы:

In [18]:
feature_cols = [col for col in train_features.columns if col != "cookie_id"]

X_train = train_features[feature_cols]
y_train = train["target"].values
X_test = test_features[feature_cols]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)

X_train: (11091, 96)
X_test: (4909, 96)
y_train: (11091,)


Обучаем выбранную конфигурацию CatBoost на всей обучающей выборке.

In [21]:
def make_best_catboost():
    return CatBoostClassifier(iterations=600, learning_rate=0.05, depth=6, l2_leaf_reg=3.0, random_seed=0, verbose=False, thread_count=-1)

In [22]:
final_catboost = make_best_catboost()
final_catboost.fit(X_train, y_train)

catboost_test_pred = final_catboost.predict_proba(X_test)[:, 1]

Аналогично обучаем лучший вариант LightGBM

In [23]:
from lightgbm import LGBMClassifier

In [24]:
def make_best_lightgbm():
    return LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=63, max_depth=-1, min_child_samples=15, subsample=0.9, colsample_bytree=0.9, reg_lambda=2.0, reg_alpha=0.1, random_state=0, n_jobs=-1, verbosity=-1)

In [25]:
final_lightgbm = make_best_lightgbm()
final_lightgbm.fit(X_train, y_train)

lightgbm_test_pred = final_lightgbm.predict_proba(X_test)[:, 1]

Итоговый скор получаем как взвешенное среднее вероятностей двух моделей.

In [26]:
test_pred = 0.6 * catboost_test_pred + 0.4 * lightgbm_test_pred

# Формируем сабмишен

In [28]:
sample_submission = pd.read_csv("sample_submission.csv")
sample_submission.head()

,cookie_id,score
0,ck_99a4e5ef02d89493,0.5
1,ck_54d463f7fd89ec3d,0.5
2,ck_0fbbc7a6af300368,0.5
3,ck_8fd4937eadd64fb6,0.5
4,ck_dbb4bd4eda97255d,0.5


In [29]:
test_predictions = pd.DataFrame({"cookie_id": test["cookie_id"], "score": test_pred})
submission = sample_submission[["cookie_id"]].merge(test_predictions, on="cookie_id", how="left")

In [31]:
submission.to_csv("submission.csv", index=False)

In [32]:
submission.head()

,cookie_id,score
0,ck_99a4e5ef02d89493,0.011906
1,ck_54d463f7fd89ec3d,0.003635
2,ck_0fbbc7a6af300368,0.001637
3,ck_8fd4937eadd64fb6,0.003664
4,ck_dbb4bd4eda97255d,0.009000


In [33]:
prediction_debug = pd.DataFrame({"cookie_id": test["cookie_id"], "catboost_score": catboost_test_pred, "lightgbm_score": lightgbm_test_pred, "blend_score": test_pred})
prediction_debug.to_csv("prediction_debug.csv", index=False)